In [43]:
# ============================================================================
# SpaceX Launch Records Dashboard - FULLY FIXED VERSION (Scatter Plot Working)
# ============================================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash
import numpy as np

print("=" * 70)
print("SPACEX DASHBOARD - INITIALIZING")
print("=" * 70)

# ============================================================================
# STEP 1: Load and Clean Data
# ============================================================================

spacex_df = pd.read_csv('data/spacex_cleaned.csv')
spacex_df['Date'] = pd.to_datetime(spacex_df['Date'])

# Clean payload column
if 'PayloadMass' not in spacex_df.columns:
    raise KeyError("❌ Column 'PayloadMass' not found in dataset! Check your CSV file.")
spacex_df['PayloadMass'] = pd.to_numeric(spacex_df['PayloadMass'], errors='coerce')

# Drop rows without payload data
spacex_df.dropna(subset=['PayloadMass'], inplace=True)

print(f"\n✅ Data loaded successfully! Shape: {spacex_df.shape}")
print(f"PayloadMass range: {spacex_df['PayloadMass'].min():,.0f} - {spacex_df['PayloadMass'].max():,.0f} kg")
print(f"Class distribution: {spacex_df['Class'].value_counts().to_dict()}")

# ============================================================================
# STEP 2: Initialize Dash Application
# ============================================================================

app = Dash(__name__)

# ============================================================================
# STEP 3: Define Layout
# ============================================================================

app.layout = html.Div([
    html.Div([
        html.H1('SpaceX Launch Records Dashboard',
                style={'textAlign': 'center', 'color': '#FFFFFF', 'padding': '20px'}),
        html.P('Interactive Dashboard for Falcon 9 Launch Analysis',
               style={'textAlign': 'center', 'color': '#FFFFFF'})
    ], style={'backgroundColor': '#1e3a5f', 'marginBottom': '30px'}),

    html.Div([
        html.Label('Select Launch Site:', style={'fontWeight': 'bold', 'fontSize': 18}),
        dcc.Dropdown(
            id='site-dropdown',
            options=[{'label': 'All Sites', 'value': 'ALL'}] +
                    [{'label': site, 'value': site} for site in sorted(spacex_df['LaunchSite'].unique())],
            value='ALL',
            placeholder="Select a Launch Site",
            searchable=True,
            style={'width': '100%', 'fontSize': 16}
        ),
    ], style={
        'width': '60%',
        'margin': '0 auto 30px auto',
        'padding': '20px',
        'backgroundColor': '#f8f9fa',
        'borderRadius': '10px'
    }),

    html.Div([
        dcc.Graph(id='success-pie-chart')
    ], style={'marginBottom': '40px'}),

    html.Div([
        html.Label('Select Payload Range (kg):', style={'fontWeight': 'bold', 'fontSize': 18}),
        dcc.RangeSlider(
            id='payload-slider',
            min=int(spacex_df['PayloadMass'].min()),
            max=int(spacex_df['PayloadMass'].max()),
            step=500,
            value=[int(spacex_df['PayloadMass'].min()), int(spacex_df['PayloadMass'].max())],
            marks={int(x): f"{int(x):,}" for x in np.linspace(spacex_df['PayloadMass'].min(),
                                                             spacex_df['PayloadMass'].max(), 6)},
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={
        'width': '85%',
        'margin': '0 auto 40px auto',
        'padding': '20px',
        'backgroundColor': '#f8f9fa',
        'borderRadius': '10px'
    }),

    html.Div([
        dcc.Graph(id='success-payload-scatter-chart')
    ], style={
        'marginBottom': '30px',
        'backgroundColor': '#ffffff',
        'padding': '20px',
        'borderRadius': '10px',
        'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'
    }),

    html.Div(id='stats-output', style={'textAlign': 'center', 'margin': '40px auto', 'fontSize': 16})
])

# ============================================================================
# STEP 4: Callback for Pie Chart
# ============================================================================

@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        site_success = spacex_df.groupby('LaunchSite')['Class'].sum().reset_index()
        fig = px.pie(
            site_success,
            names='LaunchSite',
            values='Class',
            title='Total Successful Launches by Site'
        )
    else:
        filtered_df = spacex_df[spacex_df['LaunchSite'] == selected_site]
        outcomes = filtered_df['Class'].value_counts().reset_index()
        outcomes.columns = ['Outcome', 'Count']
        outcomes['Outcome'] = outcomes['Outcome'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(outcomes, names='Outcome', values='Count',
                     title=f'Launch Outcomes for {selected_site}')
    fig.update_layout(title_x=0.5)
    return fig

# ============================================================================
# STEP 5: Callback for Scatter Plot
# ============================================================================

@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_scatter_chart(selected_site, payload_range):
    low, high = payload_range
    print(f"DEBUG: Site={selected_site}, Payload range={low}-{high}")

    # Filter by payload
    mask = (spacex_df['PayloadMass'] >= low) & (spacex_df['PayloadMass'] <= high)

    if selected_site == 'ALL':
        filtered_df = spacex_df[mask]
        title = f'Payload Mass vs Launch Success (All Sites)'
    else:
        filtered_df = spacex_df[(spacex_df['LaunchSite'] == selected_site) & mask]
        title = f'Payload Mass vs Launch Success ({selected_site})'

    print(f"DEBUG: Filtered rows = {len(filtered_df)}")

    if filtered_df.empty:
        fig = go.Figure()
        fig.add_annotation(text="<b>No data for selected filters</b>",
                           x=0.5, y=0.5, showarrow=False)
        fig.update_layout(title=title, xaxis_visible=False, yaxis_visible=False)
        return fig

    filtered_df['Outcome'] = filtered_df['Class'].map({1: 'Success', 0: 'Failure'})

    fig = px.scatter(
        filtered_df,
        x='PayloadMass',
        y='Class',
        color='Outcome',
        hover_data=['LaunchSite', 'PayloadMass', 'Date'],
        labels={'Class': 'Launch Outcome (0=Fail, 1=Success)', 'PayloadMass': 'Payload Mass (kg)'},
        title=title
    )

    fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
    fig.update_layout(title_x=0.5, yaxis=dict(tickvals=[0, 1], ticktext=['Failure', 'Success']),
                      plot_bgcolor='rgba(240,240,240,0.3)')
    return fig

# ============================================================================
# STEP 6: Callback for Statistics Panel
# ============================================================================

@app.callback(
    Output('stats-output', 'children'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_stats(selected_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['PayloadMass'] >= low) & (spacex_df['PayloadMass'] <= high)
    if selected_site == 'ALL':
        filtered = spacex_df[mask]
    else:
        filtered = spacex_df[(spacex_df['LaunchSite'] == selected_site) & mask]

    total = len(filtered)
    success = filtered['Class'].sum()
    fail = total - success
    rate = (success / total * 100) if total > 0 else 0

    return html.Div([
        html.H3("Current Selection Summary", style={'color': '#1e3a5f'}),
        html.P(f"Total Launches: {total} | Success: {success} | Failure: {fail} | Success Rate: {rate:.1f}%", 
               style={'color': '#555', 'fontSize': 18})
    ])

# ============================================================================
# STEP 7: Run Application
# ============================================================================

if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("🚀 Launching SpaceX Dashboard on http://127.0.0.1:8050/")
    print("=" * 70 + "\n")
    app.run(debug=True, port=8050)


SPACEX DASHBOARD - INITIALIZING

✅ Data loaded successfully! Shape: (179, 14)
PayloadMass range: 330 - 15,600 kg
Class distribution: {1: 144, 0: 35}

🚀 Launching SpaceX Dashboard on http://127.0.0.1:8050/

